# Trabalho — EDA Multimodal: Modalidade Texto

**Dataset:** AG News  
**Extrator:** Sentence Transformer — `all-MiniLM-L6-v2`  
**Autor:** [Seu Nome]  

---

**Lógica central:**

> Pergunta -> Representação -> Qualidade -> Exploração -> Hipótese -> Classificação

Atividade baseada na **Seção 10** do Laboratório de Ciência de Dados — EDA Multimodal.

> [AVISO] **Nota:** Ative GPU em `Ambiente de execução -> Alterar tipo -> T4 GPU` para maior velocidade.

## Seção 0 — Setup do Ambiente

In [ ]:
# Instalar dependências (necessário apenas no Colab)
!pip install -q datasets sentence-transformers plotly

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score, accuracy_score
from sklearn.ensemble import IsolationForest
from pandas.plotting import parallel_coordinates

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 50)

print('[OK] Ambiente configurado com sucesso.')

## Item 1 — Pergunta de Investigação

Os embeddings semânticos gerados pelo Sentence Transformer (`MiniLM`) conseguem capturar
diferenças temáticas entre notícias de forma que um classificador simples — como a
**Regressão Logística** — consiga separar as quatro categorias do AG News
(`World`, `Sports`, `Business`, `Sci/Tech`) com desempenho razoável?

Especificamente: categorias com linguagem mais distinta (ex. *Sports* vs *Sci/Tech*)
serão mais separáveis do que categorias com vocabulário mais próximo (ex. *Business* vs *World*)?

In [ ]:
print('=' * 65)
print('PERGUNTA DE INVESTIGAÇÃO')
print('=' * 65)
print("""
Os embeddings semânticos do MiniLM conseguem separar as
quatro categorias de notícias do AG News:

  -> World    (Mundo / Política internacional)
  -> Sports   (Esportes)
  -> Business (Negócios / Economia)
  -> Sci/Tech (Ciência e Tecnologia)

...de forma que um classificador simples (Regressão Logística)
alcance desempenho razoável sem fine-tuning do modelo?
""")
print('[OK] Item 1 — Pergunta de investigação definida.')

## Item 2 — Unidade de Análise + Carregamento dos Dados

| Elemento | Descrição |
|---|---|
| **Linha** | 1 notícia do AG News (título + corpo em inglês) |
| **Coluna** | 1 dimensão do embedding semântico (`f_000` a `f_383`) |
| **Classe** | `class_name`: World / Sports / Business / Sci/Tech |

O objetivo é verificar se as **384 dimensões** do embedding semântico
são suficientes para distinguir os quatro temas automaticamente.

In [ ]:
from datasets import load_dataset

AG_LABEL_NAMES   = ['World', 'Sports', 'Business', 'Sci/Tech']
N_TEXT_PER_CLASS = 250  # 250 x 4 classes = 1.000 amostras (mínimo exigido)

print('Carregando dataset AG News...')
ag = load_dataset('fancyzhx/ag_news')

# Amostragem estratificada
text_items = []
for label_id in range(len(AG_LABEL_NAMES)):
    subset = ag['train'].filter(lambda x: x['label'] == label_id)
    subset = subset.shuffle(seed=RANDOM_STATE).select(range(N_TEXT_PER_CLASS))
    text_items.extend([subset[i] for i in range(len(subset))])

df_raw = pd.DataFrame({
    'text' : [x['text']  for x in text_items],
    'label': [x['label'] for x in text_items],
})
df_raw['class_name'] = df_raw['label'].map(dict(enumerate(AG_LABEL_NAMES)))

print(f'Dataset carregado: {df_raw.shape[0]} notícias x {df_raw.shape[1]} colunas')
print('\nDistribuição das classes:')
display(df_raw['class_name'].value_counts().to_frame())
df_raw.head(3)

## Item 3 — Features Utilizadas (Extração de Embeddings)

Usaremos o modelo **`all-MiniLM-L6-v2`** da biblioteca Sentence Transformers
para converter cada notícia em um vetor de **384 dimensões**.

**Por que MiniLM?**
- Treinado especificamente para gerar embeddings semânticos de sentenças
- Rápido e eficiente (versão destilada do BERT)
- Captura similaridade semântica: textos com sentido parecido ficam próximos no espaço vetorial
- Estratégia: extração direta (**zero-shot**), **sem fine-tuning**

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Carregando Sentence Transformer...')
text_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

print('Gerando embeddings — aguarde...')
X_text = text_model.encode(
    df_raw['text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False
)

print(f'\nFormato dos embeddings: {X_text.shape}')
print(f'  -> {X_text.shape[0]} notícias x {X_text.shape[1]} dimensões')

text_feature_names = [f'f_{i:03d}' for i in range(X_text.shape[1])]
df_text = pd.DataFrame(X_text, columns=text_feature_names)
df_text['label']      = df_raw['label'].values
df_text['class_name'] = df_raw['class_name'].values

print('\nPrévia do DataFrame (primeiras 5 linhas, primeiras 6 features):')
display(df_text[text_feature_names[:6] + ['class_name']].head())

## Dataset Final — Estrutura e Exportação

Montagem do dataset tabular completo conforme os requisitos do enunciado:

| Coluna | Tipo | Descrição |
|---|---|---|
| `sample_id` | int | **Identificador único** da amostra |
| `f_000` a `f_383` | float | **Embeddings MiniLM** (384 dimensões) |
| `label` | int | **Rótulo** numérico (0=World, 1=Sports, 2=Business, 3=Sci/Tech) |
| `class_name` | str | **Nome da categoria** |
| `source` | str | **Origem** do dado |
| `license` | str | **Licença** de uso |
| `n_words` | int | **Metadado**: nº de palavras |
| `n_chars` | int | **Metadado**: nº de caracteres |
| `text_preview` | str | **Metadado**: trecho inicial do texto |

In [ ]:
# Metadados textuais
df_raw['n_chars'] = df_raw['text'].str.len()
df_raw['n_words'] = df_raw['text'].str.split().str.len()

# Montagem do dataset final
df_final = df_text.copy()
df_final.insert(0, 'sample_id', range(len(df_final)))
df_final['source']       = 'AG News — fancyzhx/ag_news (Hugging Face)'
df_final['license']      = 'Academic / Non-commercial use'
df_final['n_words']      = df_raw['n_words'].values
df_final['n_chars']      = df_raw['n_chars'].values
df_final['text_preview'] = df_raw['text'].str[:80].values

print(f'Shape: {df_final.shape[0]} amostras x {df_final.shape[1]} colunas')
status = '[OK] (>= 1.000)' if df_final.shape[0] >= 1000 else '[ERRO] (< 1.000 — INSUFICIENTE)'
print(f'  -> {df_final.shape[0]} amostras {status}')

# Prévia das colunas de metadados
meta_cols = ['sample_id','label','class_name','source','license','n_words','n_chars']
display(df_final[meta_cols].head(5))

# Exportação para CSV
csv_path = 'agnews_embeddings_1000.csv'
df_final.to_csv(csv_path, index=False, encoding='utf-8')
print(f'\n[OK] Dataset exportado: {csv_path}')
print(f'   {df_final.shape[0]} linhas x {df_final.shape[1]} colunas')

# No Colab: download automático do arquivo
try:
    from google.colab import files
    files.download(csv_path)
    print('[OK] Download iniciado automaticamente.')
except ImportError:
    print(f'Arquivo salvo localmente: {csv_path}')


## Item 4 — Análise de Qualidade dos Embeddings

Verificamos quatro tipos de problema:

| Verificação | Método |
|---|---|
| Valores ausentes | `isna().sum()` |
| Duplicatas | `duplicated()` |
| Balanceamento | `value_counts()` |
| Outliers multivariados | `IsolationForest` |

> **Nota:** NaN em embeddings quase sempre indica problema no **pipeline** de processamento, não no fenômeno real.

In [ ]:
print('=' * 60)
print('ITEM 4 — ANÁLISE DE QUALIDADE')
print('=' * 60)

# 4.1 Valores ausentes
print('\n[4.1] Valores ausentes nos embeddings:')
total_missing = df_text[text_feature_names].isna().sum().sum()
print(f'  -> {total_missing} valores ausentes' if total_missing > 0 else '  [OK] Nenhum valor ausente.')

# 4.2 Duplicatas
print('\n[4.2] Duplicatas:')
n_dup = df_text[text_feature_names].duplicated().sum()
n_dup_txt = df_raw['text'].duplicated().sum()
print(f'  Vetores duplicados : {n_dup}')
print(f'  Textos duplicados  : {n_dup_txt}')

# 4.3 Balanceamento
print('\n[4.3] Balanceamento das classes:')
contagem = df_text['class_name'].value_counts()
display(contagem.to_frame())

fig, ax = plt.subplots(figsize=(7, 4))
contagem.plot(kind='bar', ax=ax,
              color=['#4e79a7','#f28e2b','#e15759','#76b7b2'],
              edgecolor='black')
ax.set_title('Distribuição das Classes — AG News (amostra)', fontweight='bold')
ax.set_xlabel('Categoria')
ax.set_ylabel('Nº de Notícias')
ax.set_xticklabels(contagem.index, rotation=20)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

# 4.4 Outliers
print('\n[4.4] Outliers — Isolation Forest (contamination=5%):')
X_scaled_qlt = StandardScaler().fit_transform(X_text)
iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
pred_iso = iso.fit_predict(X_scaled_qlt)
scores   = iso.decision_function(X_scaled_qlt)
n_out    = (pred_iso == -1).sum()
print(f'  Candidatos a outlier: {n_out} de {len(pred_iso)} ({n_out/len(pred_iso)*100:.1f}%)')

df_iso = df_raw.copy()
df_iso['anomaly_score'] = scores
df_iso['outlier']       = pred_iso
print('\n  Top 5 outliers mais extremos:')
display(df_iso[df_iso['outlier']==-1]
        .sort_values('anomaly_score')
        .head(5)[['class_name','anomaly_score','text']]
        .assign(text=lambda d: d['text'].str[:120] + '...'))

print('\n[OK] Item 4 — Análise de qualidade concluída.')

## Item 5 — Visualizações

**5a.** Comprimento dos textos por categoria (univariada)  
**5b.** Norma L2 dos embeddings por categoria (univariada)  
**5c.** Parallel Coordinates nos 8 primeiros PCs (multivariada)

In [ ]:
print('=' * 60)
print('ITEM 5 — VISUALIZAÇÕES')
print('=' * 60)

X_text_scaled = StandardScaler().fit_transform(X_text)

# 5a — Comprimento dos textos
print('\n[5a] Comprimento dos textos por categoria:')
df_raw['n_chars'] = df_raw['text'].str.len()
df_raw['n_words'] = df_raw['text'].str.split().str.len()
display(df_raw.groupby('class_name')[['n_chars','n_words']].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cls, grp in df_raw.groupby('class_name'):
    axes[0].hist(grp['n_chars'], bins=20, alpha=0.55, label=cls)
    axes[1].hist(grp['n_words'], bins=20, alpha=0.55, label=cls)
for ax, titulo, xlabel in zip(axes,
    ['Nº de caracteres por categoria', 'Nº de palavras por categoria'],
    ['Nº de caracteres', 'Nº de palavras']):
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Frequência')
    ax.legend()
plt.tight_layout()
plt.show()

# 5b — Norma L2
print('\n[5b] Norma L2 dos embeddings por categoria:')
normas    = np.linalg.norm(X_text, axis=1)
df_normas = pd.DataFrame({'norma_l2': normas, 'class_name': df_raw['class_name'].values})
fig, ax   = plt.subplots(figsize=(9, 5))
df_normas.boxplot(column='norma_l2', by='class_name', ax=ax,
                  patch_artist=True,
                  boxprops=dict(facecolor='lightblue'),
                  medianprops=dict(color='red', linewidth=2))
ax.set_title('Norma L2 dos embeddings por categoria', fontweight='bold')
ax.set_xlabel('Categoria')
ax.set_ylabel('Norma L2')
plt.suptitle('')
plt.tight_layout()
plt.show()

# 5c — Parallel Coordinates (8 PCs)
print('\n[5c] Parallel Coordinates — 8 primeiros PCs:')
pca_vis = PCA(n_components=8, random_state=RANDOM_STATE)
X_pca8  = pca_vis.fit_transform(X_text_scaled)
df_pc8  = pd.DataFrame(X_pca8, columns=[f'PC{i+1}' for i in range(8)])
df_pc8['class_name'] = df_raw['class_name'].values
parts = []
for cn, grp in df_pc8.groupby('class_name'):
    parts.append(grp.sample(min(len(grp), 12), random_state=RANDOM_STATE))
sample_pc8 = pd.concat(parts).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(13, 5))
parallel_coordinates(sample_pc8, 'class_name',
                     color=['#4e79a7','#f28e2b','#e15759','#76b7b2'],
                     alpha=0.45, linewidth=1.2, ax=ax)
ax.set_title('AG News + MiniLM — Parallel Coordinates (8 PCs)', fontweight='bold', fontsize=13)
ax.set_ylabel('Valor do componente')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print(f'  Variância explicada pelos 8 PCs: {pca_vis.explained_variance_ratio_.sum()*100:.1f}%')
print('\n[OK] Item 5 — Visualizações concluídas.')

## Item 6 — Projeção PCA 2D

Reduzimos os embeddings de **384 -> 2 dimensões** para visualizar graficamente a separação.

> [AVISO] **Atenção:** PCA 2D preserva apenas uma fração da variância original.
> Um espaço pouco separável em 2D pode ainda ser **muito separável em alta dimensão**.

In [ ]:
print('=' * 60)
print('ITEM 6 — PROJEÇÃO PCA 2D')
print('=' * 60)

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d   = pca_2d.fit_transform(X_text_scaled)
var1, var2 = pca_2d.explained_variance_ratio_ * 100
var_total  = var1 + var2

print(f'\nVariância explicada:')
print(f'  PC1        : {var1:.2f}%')
print(f'  PC2        : {var2:.2f}%')
print(f'  Total (2D) : {var_total:.2f}%')
print(f'  Oculta     : {100-var_total:.1f}% permanece invisível na projeção 2D')

cores_cat = {'World':'#4e79a7','Sports':'#f28e2b','Business':'#e15759','Sci/Tech':'#76b7b2'}
fig, ax = plt.subplots(figsize=(9, 7))
for lid, cls in enumerate(AG_LABEL_NAMES):
    mask = df_raw['label'].values == lid
    ax.scatter(X_2d[mask,0], X_2d[mask,1],
               alpha=0.60, s=40, color=cores_cat[cls],
               label=cls, edgecolors='white', linewidths=0.3)
ax.set_xlabel(f'PC1 ({var1:.1f}% da variância)', fontsize=11)
ax.set_ylabel(f'PC2 ({var2:.1f}% da variância)', fontsize=11)
ax.set_title(f'AG News + MiniLM — PCA 2D\n(variância total: {var_total:.1f}%)',
             fontsize=13, fontweight='bold')
ax.legend(title='Categoria', fontsize=10)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print('\n[OK] Item 6 — PCA 2D concluído.')

## Item 7 — Hipótese sobre Separabilidade

Com base nas visualizações (PCA 2D + Parallel Coordinates), formulamos as hipóteses
**antes** de rodar o classificador:

| Hipótese | Descrição |
|---|---|
| **H1** | *Sports* será a categoria mais fácil de separar (vocabulário muito específico) |
| **H2** | *Business* e *World* terão maior sobreposição (vocabulário compartilhado) |
| **H3** | F1 macro >= 0.80 (embeddings MiniLM suficientemente discriminativos sem fine-tuning) |

**Ordem esperada de separabilidade:** Sports > Sci/Tech > World ≈ Business

In [ ]:
print('Hipóteses formuladas antes da classificação:')
print('  H1: Sports -> mais separável (vocabulário específico)')
print('  H2: Business/World -> maior confusão (vocabulário próximo)')
print('  H3: F1 macro >= 0.80')
print('\n[OK] Item 7 — Hipótese formulada.')

## Item 8 — Teste com Regressão Logística

Usamos a **Regressão Logística** como classificador simples para testar se as
representações contêm informação discriminativa suficiente.

A complexidade baixa é **intencional**: se um modelo linear já consegue bom desempenho,
as representações são de alta qualidade.

- Split: **75% treino / 25% teste** (estratificado)

In [ ]:
print('=' * 60)
print('ITEM 8 — REGRESSÃO LOGÍSTICA')
print('=' * 60)

y_text = df_raw['label'].values
X_train, X_test, y_train, y_test = train_test_split(
    X_text_scaled, y_text,
    test_size=0.25, stratify=y_text, random_state=RANDOM_STATE
)
print(f'\nSplit: {len(X_train)} treino / {len(X_test)} teste (75/25, estratificado)\n')

clf    = LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('-' * 60)
print('CLASSIFICATION REPORT')
print('-' * 60)
print(classification_report(y_test, y_pred, target_names=AG_LABEL_NAMES))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=AG_LABEL_NAMES,
    xticks_rotation=30, ax=ax, colorbar=False
)
ax.set_title('AG News + MiniLM — Regressão Logística\nMatriz de Confusão', fontweight='bold')
plt.tight_layout()
plt.show()

f1_macro = f1_score(y_test, y_pred, average='macro')
acc      = accuracy_score(y_test, y_pred)
print(f'Acurácia geral : {acc*100:.1f}%')
print(f'F1 macro       : {f1_macro:.4f}')
print('\n- Comparação com as Hipóteses -')
print('  H3:', '[OK] CONFIRMADA' if f1_macro >= 0.80 else '[ERRO] NÃO CONFIRMADA', f'(F1={f1_macro:.2f})')
print('  -> Verifique H1 e H2 no classification_report acima.')
print('\n[OK] Item 8 — Classificação concluída.')

## Item 9 — Limitações da Análise

| # | Limitação |
|---|---|
| 1 | **Tamanho da amostra:** 120/classe (480 total) de 120.000+ textos disponíveis |
| 2 | **PCA 2D:** captura apenas ~7% da variância — pode induzir conclusões erradas |
| 3 | **Modelo sem fine-tuning:** MiniLM foi treinado para similaridade geral, não para AG News |
| 4 | **Outliers não removidos:** a decisão de manter pode influenciar o classificador |
| 5 | **Classificador linear:** Regressão Logística é intencionalmente simples — modelos mais complexos provavelmente obteriam maior acurácia |

In [ ]:
print('[OK] Item 9 — Limitações documentadas na célula markdown acima.')

## Extensão Opcional — Impacto do Label Noise

Inserimos ruído nos rótulos de **10% das amostras de treino** (trocando a label por
uma categoria aleatória diferente) e medimos o impacto na acurácia.

Isso simula **erros de rotulação** que ocorrem em projetos reais.

In [ ]:
print('=' * 60)
print('EXTENSÃO — IMPACTO DO LABEL NOISE')
print('=' * 60)

# Baseline (labels limpas)
clf_clean = LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
clf_clean.fit(X_train, y_train)
acc_clean = accuracy_score(y_test, clf_clean.predict(X_test))
f1_clean  = f1_score(y_test, clf_clean.predict(X_test), average='macro')

# Inserindo label noise
NOISE_RATE    = 0.10
y_train_noisy = y_train.copy()
n_noisy       = int(len(y_train) * NOISE_RATE)
rng           = np.random.default_rng(RANDOM_STATE)
noisy_idx     = rng.choice(len(y_train), size=n_noisy, replace=False)
for idx in noisy_idx:
    orig = y_train_noisy[idx]
    y_train_noisy[idx] = rng.choice([l for l in range(len(AG_LABEL_NAMES)) if l != orig])

print(f'Noise rate: {NOISE_RATE*100:.0f}% ({n_noisy} de {len(y_train)} amostras com label trocada)\n')

# Treino com ruído
clf_noisy = LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
clf_noisy.fit(X_train, y_train_noisy)
acc_noisy = accuracy_score(y_test, clf_noisy.predict(X_test))
f1_noisy  = f1_score(y_test, clf_noisy.predict(X_test), average='macro')

# Tabela comparativa
df_comp = pd.DataFrame({
    'Métrica'    : ['Acurácia (%)', 'F1 Macro'],
    'Sem Ruído'  : [f'{acc_clean*100:.1f}', f'{f1_clean:.4f}'],
    'Com Ruído (10%)': [f'{acc_noisy*100:.1f}', f'{f1_noisy:.4f}'],
    'Queda'      : [f'{(acc_clean-acc_noisy)*100:.1f} pp', f'{f1_clean-f1_noisy:.4f}']
})
display(df_comp)

# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, clf_m, titulo in zip(axes,
    [clf_clean, clf_noisy],
    [f'Sem label noise\nAcc={acc_clean*100:.1f}%  F1={f1_clean:.2f}',
     f'Com label noise (10%)\nAcc={acc_noisy*100:.1f}%  F1={f1_noisy:.2f}']):
    ConfusionMatrixDisplay.from_predictions(
        y_test, clf_m.predict(X_test),
        display_labels=AG_LABEL_NAMES,
        xticks_rotation=30, ax=ax, colorbar=False
    )
    ax.set_title(titulo, fontweight='bold')
plt.suptitle('Impacto do Label Noise — AG News + MiniLM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n[OK] Extensão — Label Noise concluída.')

## [OK] Trabalho Concluído

| Item | Status |
|---|---|
| 1. Pergunta de investigação | [OK] |
| 2. Unidade de análise | [OK] |
| 3. Features utilizadas (MiniLM 384-dim) | [OK] |
| 4. Análise de qualidade | [OK] |
| 5. Visualizações (comprimento, norma L2, Parallel Coords) | [OK] |
| 6. Projeção PCA 2D | [OK] |
| 7. Hipótese sobre separabilidade | [OK] |
| 8. Regressão Logística + métricas | [OK] |
| 9. Limitações da análise | [OK] |
| Extensão: Label Noise | [OK] |